# SkinGAN — full pipeline on Google Colab

**SkinGAN = conditional StyleGAN2-ADA + a 7-class classifier that gives feedback to the generator.**
This notebook runs the whole experiment from the paper:

| Step | What | T4 time (estimate) |
|---|---|---|
| **A** | Train the domain classifier (DenseNet-121 on real images) | ~20–40 min |
| **B** | **Standard GAN**: continue from the shared checkpoint for `EXTRA_KIMG` | ~7 h for 300 kimg |
| **C** | **SkinGAN**: same checkpoint, same `EXTRA_KIMG`, classifier loss ON | ~8–9 h for 300 kimg |
| **D** | Generate 1,000 synthetic images per class from each GAN | ~10 min |
| **E** | Classification experiment: real / traditional aug / +GAN / +SkinGAN / SkinGAN only | ~1 h (1 seed) |
| **F** | Report + comparison pictures | ~5 min |

Both GANs start from the **same checkpoint** (your finished `baseline_test` run) and train for the **same
kimg**, so the only difference between them is the classifier loss.

**Every session:** `Runtime → Change runtime type → T4 GPU`, then run cells **0–5**, then the next unfinished step.
Each step **skips itself when done** and **resumes after a disconnect**, so you can simply run everything again.
Results live in Google Drive under `MyDrive/SkinGAN/skingan_outputs/`.

## 0 · Settings

In [ ]:
REPO_URL   = "https://github.com/AmoghNamuduri/SkinLesion.git"
BRANCH     = "main"
DRIVE_DIR  = "/content/drive/MyDrive/SkinGAN"
EXTRA_KIMG = 300      # kimg each GAN trains beyond the shared checkpoint (lower = cheaper, e.g. 200)
SNAP       = 10       # snapshot every 10 ticks = 40 kimg
LAMBDA     = 0.1      # weight of the classifier loss in SkinGAN (SpineGAN used 0.1)
SEEDS      = [0]      # classifier seeds in step E; [0, 1] if you have time
START_FROM = f"{DRIVE_DIR}/runs/baseline_test"   # the run that provides the shared checkpoint

## 1 · GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Drive, code, paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, shutil
os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.exists('/content/SkinLesion'):
    !git clone -q -b {BRANCH} {REPO_URL} /content/SkinLesion
else:
    !git -C /content/SkinLesion pull -q
%cd /content/SkinLesion
!pip -q install ninja click
sys.path.insert(0, '/content/SkinLesion/src')
from skinlesion.gan.train_gan import latest_snapshot, kimg_done, resume_plan, prune_snapshots

OUT       = f"{DRIVE_DIR}/skingan_outputs"          # everything SkinGAN produces
ZIP_NAME  = "ham10000_256_7class.zip"
DRIVE_ZIP = f"{DRIVE_DIR}/data/{ZIP_NAME}"
LOCAL_ZIP = f"/content/data/{ZIP_NAME}"
SPLITS    = f"{DRIVE_DIR}/data/splits"
RAW       = "/content/ham_raw"
CLF       = f"{OUT}/domain_classifier/domain_classifier.pt"
SHARED    = f"{DRIVE_DIR}/checkpoints/shared_start.pkl"
RUNS_GAN  = f"{OUT}/gan/baseline"
RUNS_SKIN = f"{OUT}/gan/skingan"
FIGS      = f"{OUT}/figures"
SEEDS_STR = str(SEEDS).replace(" ", "")
COMMON = (f"paths.raw_dir={RAW} paths.splits_dir={SPLITS} paths.gan_dataset={LOCAL_ZIP} "
          f"paths.outputs_dir={OUT} paths.domain_classifier={CLF}")
for d in (OUT, FIGS, os.path.dirname(SHARED), "/content/data"):
    os.makedirs(d, exist_ok=True)

## 3 · StyleGAN2-ADA + SkinGAN patches

In [ ]:
!python third_party/setup_stylegan.py

## 4 · Data
The 256² training archive comes from Drive (built by the test notebook). `ensure_raw()` downloads the original
HAM10000 images (~5 min); only steps **A**, **E** and the memorization check need them, and they call it themselves.

In [ ]:
if os.path.exists(DRIVE_ZIP):
    shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
else:
    !python scripts/download_data.py --set paths.raw_dir={RAW}
    !python scripts/export_gan_dataset.py --set paths.raw_dir={RAW} paths.gan_dataset={LOCAL_ZIP} paths.splits_dir={SPLITS}
    os.makedirs(os.path.dirname(DRIVE_ZIP), exist_ok=True); shutil.copy(LOCAL_ZIP, DRIVE_ZIP)
print("training archive:", round(os.path.getsize(LOCAL_ZIP) / 1e6), "MB")

def ensure_raw():
    if glob.glob(f"{RAW}/**/HAM10000_metadata.csv", recursive=True) and len(glob.glob(f"{RAW}/**/*.jpg", recursive=True)) > 10000:
        return print("HAM10000 images ready")
    !python scripts/download_data.py --set paths.raw_dir={RAW}
    for z in glob.glob(f"{RAW}/*.zip"):
        os.remove(z)   # already extracted; free disk space

## 5 · Shared starting checkpoint
Copies the newest snapshot of your test run to `checkpoints/shared_start.pkl` **once**. Both GANs start from
this exact file. (If there is no test run, both start from NVIDIA's FFHQ-256 model instead.)

In [ ]:
if not os.path.exists(SHARED):
    src = latest_snapshot(START_FROM) if os.path.exists(START_FROM) else None
    if src:
        shutil.copy(src, SHARED)
        json.dump({"source": str(src), "kimg_before": kimg_done(START_FROM)},
                  open(f"{os.path.dirname(SHARED)}/shared_start.json", "w"), indent=2)
START = SHARED if os.path.exists(SHARED) else "ffhq256"
info = f"{os.path.dirname(SHARED)}/shared_start.json"
print("both GANs start from:", START)
if os.path.exists(info): print(open(info).read())

## A · Domain classifier
DenseNet-121 (ImageNet) trained on the **real** training split, best epoch chosen on validation AUC.
It is frozen and used inside SkinGAN's loss. Skipped if it already exists on Drive.

In [ ]:
if os.path.exists(CLF):
    print("domain classifier already trained:", CLF)
else:
    ensure_raw()
    !python scripts/train_domain_classifier.py --config skinGan/colab_skingan.yaml --set {COMMON}
r = json.load(open(f"{OUT}/domain_classifier/results.json"))
print(f"test AUC {r['test']['auc']:.3f}  (95% CI {r['test']['auc_ci']['low']:.3f}-{r['test']['auc_ci']['high']:.3f}), "
      f"balanced accuracy {r['test']['balanced_accuracy']:.3f}")

## B · Standard GAN (no classifier feedback)
Continues from the shared checkpoint for `EXTRA_KIMG`. **Re-run this cell after a disconnect**: it resumes
from the newest snapshot and trains only what is missing. Leave the tab open while it runs.

In [ ]:
plan = resume_plan(RUNS_GAN, START, EXTRA_KIMG); print(plan)
if not plan["finished"]:
    RESUME, KIMG = plan["resume"], plan["remaining"]
    !python scripts/train_gan.py --config skinGan/colab_baseline.yaml --set {COMMON} paths.gan_runs={RUNS_GAN} gan.resume={RESUME} gan.kimg={KIMG} gan.snap={SNAP}
print("standard GAN kimg done:", kimg_done(RUNS_GAN), "/", EXTRA_KIMG)

## C · SkinGAN (classifier feedback ON)
Same checkpoint, same kimg, plus `LAMBDA · CrossEntropy(classifier(G(z, c)), c)` in the generator loss.
The log should say **`+ SkinGAN domain loss`** and **`[SkinGAN] domain loss on`**. Re-run after a disconnect.

In [ ]:
assert os.path.exists(CLF), "run step A first"
plan = resume_plan(RUNS_SKIN, START, EXTRA_KIMG); print(plan)
if not plan["finished"]:
    RESUME, KIMG = plan["resume"], plan["remaining"]
    !python scripts/train_gan.py --config skinGan/colab_skingan.yaml --set {COMMON} paths.gan_runs={RUNS_SKIN} gan.resume={RESUME} gan.kimg={KIMG} gan.snap={SNAP} gan.domain_loss.weight={LAMBDA}
print("SkinGAN kimg done:", kimg_done(RUNS_SKIN), "/", EXTRA_KIMG)

### Optional · free Drive space (keeps the 2 newest snapshots per GAN)

In [ ]:
for runs in (RUNS_GAN, RUNS_SKIN):
    removed = prune_snapshots(runs, keep=2) if os.path.exists(runs) else []
    print(runs, "-> removed", len(removed), "old snapshots")

## D · Synthetic images
1,000 images per class from each GAN's newest snapshot (same seeds for both), into `/content/synthetic/<gan>`.
They are regenerated each session (fast), so they don't fill your Drive.

In [ ]:
for arm, runs, cfg in (("baseline", RUNS_GAN, "skinGan/colab_baseline.yaml"), ("skingan", RUNS_SKIN, "skinGan/colab_skingan.yaml")):
    if os.path.exists(f"/content/synthetic/{arm}/manifest.csv"):
        print(arm, "already generated"); continue
    !python scripts/generate_synthetic.py --config {cfg} --set {COMMON} paths.gan_runs={runs} paths.synthetic_dir=/content/synthetic/{arm} generate.policy=balanced generate.per_class=1000
    shutil.copy(f"/content/synthetic/{arm}/sample_grid.png", f"{FIGS}/samples_{arm}.png")

## E · Classification experiment (the paper's results)
The same DenseNet-121 is trained on each training set and tested on the **same real test images**:
`real` · `traditional_aug` (rare classes topped up to 1,000 with copies + flips/rotations/crops) ·
`real_plus_gan` · `real_plus_skingan` (topped up with synthetic images) · `skingan_only` (SpineGAN-style).

In [ ]:
ensure_raw()
!python scripts/run_synthetic_learning.py --config skinGan/colab_experiment.yaml --set {COMMON} synthetic_learning.seeds={SEEDS_STR}

## F · Report and pictures

In [ ]:
!python skinGan/compare_grids.py --data {LOCAL_ZIP} --baseline-runs {RUNS_GAN} --skingan-runs {RUNS_SKIN} --out {FIGS}
from IPython.display import Image, Markdown, display
for name in ["real_vs_gan_vs_skingan.png", "progression_baseline.png", "progression_skingan.png"]:
    if os.path.exists(f"{FIGS}/{name}"):
        print(name); display(Image(f"{FIGS}/{name}", width=1000))
if os.path.exists(f"{OUT}/report-skingan.md"):
    display(Markdown(open(f"{OUT}/report-skingan.md").read()))

### Optional · FID for both GANs (~15 min each) and the memorization check
Lower FID = generated images closer to the real ones. The memorization check confirms SkinGAN makes new
lesions instead of copying training images. Both are added to the report.

In [ ]:
for runs in (RUNS_GAN, RUNS_SKIN):
    pkl = latest_snapshot(runs)
    !cd third_party/stylegan2-ada-pytorch && PYTHONWARNINGS=ignore python calc_metrics.py --metrics=fid10k_full --network={pkl} --data={LOCAL_ZIP} --mirror=1
ensure_raw()
!python scripts/audit_memorization.py --config skinGan/colab_skingan.yaml --set {COMMON} paths.synthetic_dir=/content/synthetic/skingan
for f in ("memorization.json", "closest_pairs.png"):
    shutil.copy(f"/content/synthetic/skingan/{f}", f"{FIGS}/{f}")
sys.path.insert(0, 'src')
from skinlesion.evaluation.report import build_skingan_report
print(build_skingan_report(OUT))
display(Markdown(open(f"{OUT}/report-skingan.md").read()))